## Load the dataset

In [9]:
from langchain_neo4j import Neo4jGraph
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
from config import NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, GROQ_API_KEY

# Neo 4J support
graph = Neo4jGraph(
    url = NEO4J_URI,
    username = NEO4J_USERNAME,
    password = NEO4J_PASSWORD
)

llm = ChatGroq(
    groq_api_key = GROQ_API_KEY,
    model = "openai/gpt-oss-120b"
)

In [10]:
movie_query = """
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

# execuuting the query in our neo4j database
graph.query(movie_query)

[]

In [11]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, title: STRING, released: DATE, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


# Using LLM to generate cypher query and retrieve relevant content from the dataset

## GraphQAChain has been deprecated. Now we need to create a custom tool to execute cypher query

In [13]:
from langchain.agents import create_agent
from langchain.tools import tool

# Creating a custom tool to retrieve results from neo4j
@tool

def query_tool(cypher_query):
    """Execute a Cypher query on the Neo4j graph."""
    result = graph.query(cypher_query)
    return str(result)

agent = create_agent(
    model = llm,
    tools = [query_tool],
    system_prompt = "You are a helpful assistant that can query a knowledge graph. Generate Cypher queries to answer questions."
)

result = agent.invoke({
    "messages" : [{"role" : "user", "content" : "Which movie has the most imdb rating?"}]
})

print(result["messages"][-1].content)

The movie with the highest IMDb rating in the database is **“The Shawshank Redemption”**, which has an IMDb rating of **9.3**.
